# The words Excellent, Good, Typical, Fair, Poor

Several columns share one ordered vocabulary: `Ex`, `Gd`, `TA`, `Fa`, `Po`. `TA` means typical, the ordinary grade. Kitchen quality and exterior quality use that vocabulary, and neither one uses `Po` in the training file.

Kitchen quality:

| Word | Houses | Mean price |
|---|---:|---:|
| `Fa` | 39 | about 105565 |
| `TA` | 735 | about 139963 |
| `Gd` | 586 | about 212116 |
| `Ex` | 100 | about 328555 |

Exterior quality:

| Word | Houses | Mean price |
|---|---:|---:|
| `Fa` | 14 | about 87985 |
| `TA` | 906 | about 144341 |
| `Gd` | 488 | about 231634 |
| `Ex` | 52 | about 367361 |

The order `Fa`, `TA`, `Gd`, `Ex` agrees with the mean price in both columns. That is a reason to keep the order, and it is not yet a reason to code them as 1, 2, 3, 4 and fit a line. The gaps are uneven: kitchen `TA` to `Gd` is a larger step than `Fa` to `TA`. House 1 has `Gd` on both the kitchen and the exterior.

`OverallQual` is already the numeric summary of this kind of judgment. The words are the same idea, written on one part of the house at a time.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# Mean price for each quality word, kitchen and exterior.
from collections import defaultdict
def means(column):
    groups = defaultdict(list)
    for row in train:
        groups[row[column]].append(int(row["SalePrice"]))
    return groups

kitchen = means("KitchenQual")
exterior = means("ExterQual")
kitchen_n = {"Fa": 39, "TA": 735, "Gd": 586, "Ex": 100}
exterior_n = {"Fa": 14, "TA": 906, "Gd": 488, "Ex": 52}
for word, count in kitchen_n.items():
    print("kitchen", word, len(kitchen[word]), sum(kitchen[word]) / count)
    assert len(kitchen[word]) == count
for word, count in exterior_n.items():
    print("exterior", word, len(exterior[word]), sum(exterior[word]) / count)
    assert len(exterior[word]) == count
order = ["Fa", "TA", "Gd", "Ex"]
kitchen_means = [sum(kitchen[word]) / kitchen_n[word] for word in order]
exterior_means = [sum(exterior[word]) / exterior_n[word] for word in order]
assert kitchen_means == sorted(kitchen_means)
assert exterior_means == sorted(exterior_means)
assert "Po" not in kitchen and "Po" not in exterior
assert train[0]["KitchenQual"] == "Gd"
assert train[0]["ExterQual"] == "Gd"


kitchen Fa 39 105565.20512820513
kitchen TA 735 139962.51156462586
kitchen Gd 586 212116.02389078497
kitchen Ex 100 328554.67
exterior Fa 14 87985.21428571429
exterior TA 906 144341.31346578366
exterior Gd 488 231633.51024590165
exterior Ex 52 367360.96153846156


## Pitfall

The means rise in the order the words suggest. The steps are not equal, and `Po` does not appear. Numbering the words 1 through 4 invents a spacing the table has not earned.

## Summary

Kitchen and exterior quality use `Fa`, `TA`, `Gd`, and `Ex`. In both columns the mean price rises in that order. Typical kitchens are 735 houses. Excellent kitchens are 100.
